In [ ]:
# !ls -lh /kaggle/input/datasets/hftams/zebrahub-zsns003

In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input/datasets/hftams/zebrahub-zsns003'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [ ]:
import copy
import hashlib
import json
import shutil
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import copy
import torch
import torch.nn.functional as functional

import time

input_root = Path("/kaggle/input")

# 新模型包上传到 Kaggle 后，修改这一处即可。
# 第一次使用原始模型时改成：
# SECONDARY_SLUG = "biohub-temporal-unet3d-seed314159-v1"
SECONDARY_SLUG = "biohub-temporal-unet3d-seed314159-v1"


def choose_existing_dir(name, candidates):
    for path in candidates:
        if path.is_dir():
            print(f"{name}: {path}", flush=True)
            return path
    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"{name} not found. Checked:\n{checked}"
    )


path_t0 = time.perf_counter()

support = choose_existing_dir(
    "support",
    [
        input_root / "datasets" / "pilkwang"
        / "biohub-tracking-support-pack-50ep-v1",
        input_root / "datasets" / "hftams"
        / "biohub-tracking-support-pack-50ep-v1",
        input_root / "biohub-tracking-support-pack-50ep-v1",
    ],
)

secondary = choose_existing_dir(
    "secondary",
    [
        input_root / "datasets" / "pilkwang" / SECONDARY_SLUG,
        input_root / "datasets" / "hftams" / SECONDARY_SLUG,
        input_root / SECONDARY_SLUG,
    ],
)

competition = choose_existing_dir(
    "competition",
    [
        input_root / "competitions"
        / "biohub-cell-tracking-during-development",
        input_root / "biohub-cell-tracking-during-development",
    ],
)

assert (competition / "train").is_dir(), (
    f"train directory not found: {competition / 'train'}"
)

print(
    f"Path initialization: {time.perf_counter() - path_t0:.2f}s",
    flush=True,
)

run_dir = Path(tempfile.mkdtemp(prefix="secondary_ft_", dir="/kaggle/working"))
repo_dir = run_dir / "repo"
initial_weights_root = run_dir / "initial_weights"

print("support:", support)
print("secondary:", secondary)
print("competition:", competition)
print("run_dir:", run_dir)

# 只安装训练脚本实际需要的顶层包。
# --no-deps 是关键：避免 tracksdata 自动拉取 imagecodecs==2026.6.26。
runtime_packages = [
    "tracksdata",
    "polars==1.42.0",
    "polars-runtime-32==1.42.0",
    "zarr==3.2.1",
    "geff==1.2.0.1.1",
    "geff-spec==1.1.1",
    "numcodecs==0.15.1",
    "donfig==0.8.1.post1",
    "bidict==0.23.1",
    "ilpy==0.6.0",
    "pyscipopt==6.2.1",
    "dask",
    "blosc2",
    "pyarrow",
    "rustworkx",
    "sqlalchemy",
]

wheels_dir = support / "wheels"
assert wheels_dir.is_dir(), f"wheels directory not found: {wheels_dir}"

install_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-index",
    "--no-deps",
    "--force-reinstall",
    "--find-links",
    str(wheels_dir),
    *runtime_packages,
]

print("Installing runtime packages with --no-deps...")
print(" ".join(install_command))
subprocess.run(install_command, check=True)

# 复制推理仓库源码。
repo_source = support / "repo"
repo_archive = support / "repo.zip"

if repo_source.is_dir():
    shutil.copytree(repo_source, repo_dir)
elif repo_archive.is_file():
    shutil.unpack_archive(str(repo_archive), str(run_dir))
    if not repo_dir.exists():
        extracted = next(
            path for path in run_dir.iterdir()
            if path.is_dir() and path.name not in {"initial_weights"}
        )
        extracted.rename(repo_dir)
else:
    raise FileNotFoundError("Support pack contains neither repo/ nor repo.zip")

# 复制副模型权重。
secondary_weights_source = secondary / "weights"
secondary_weights_archive = secondary / "weights.zip"

if secondary_weights_source.is_dir():
    shutil.copytree(secondary_weights_source, initial_weights_root)
elif secondary_weights_archive.is_file():
    shutil.unpack_archive(str(secondary_weights_archive), str(initial_weights_root))
else:
    raise FileNotFoundError("Secondary artifact contains neither weights/ nor weights.zip")

secondary_weights = (
    initial_weights_root
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)
secondary_config = secondary_weights.with_name("config.json")

assert secondary_weights.is_file(), secondary_weights
assert secondary_config.is_file(), secondary_config

sys.path[:0] = [
    str(repo_dir / "scripts"),
    str(repo_dir / "src"),
]

train_script = repo_dir / "scripts" / "train_unet_transformer.py"
train_source = train_script.read_text(encoding="utf-8")


def replace_once(source, old, new, label):
    count = source.count(old)
    assert count == 1, (
        f"{label}: expected exactly one match, found {count}"
    )
    return source.replace(old, new, 1)


# 1. 给 train() 增加 validate 参数。
train_source = replace_once(
    train_source,
    """    pool_kernel_um: float = 5.0,
    data_parallel: bool = True,
) -> UNetNodeTransformer:""",
    """    pool_kernel_um: float = 5.0,
    data_parallel: bool = True,
    validate: bool = True,
) -> UNetNodeTransformer:""",
    "train signature",
)

# 3. validate=False 时不加载验证数据。
train_source = replace_once(
    train_source,
    """    train_video_data = _load(train_files, "train")
    test_video_data = _load(test_files, "test")
""",
    """    train_video_data = _load(train_files, "train")
    test_video_data = _load(test_files, "test") if validate else []
""",
    "validation data loading",
)


# 4. 不创建验证 Dataset。
train_source = replace_once(
    train_source,
    "    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes)\n",
    "    test_ds = FrameWindowDataset(test_video_data, max_nodes=max_nodes) if validate else None\n",
    "validation dataset",
)


# 5. 不创建验证 DataLoader。
train_source = replace_once(
    train_source,
    """    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0, pin_memory=False,
        generator=g, worker_init_fn=worker_init_fn,
    )
""",
    """    test_loader = (
        DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            prefetch_factor=2 if num_workers > 0 else None,
            persistent_workers=num_workers > 0,
            pin_memory=False,
            generator=g,
            worker_init_fn=worker_init_fn,
        )
        if validate
        else None
    )
""",
    "validation dataloader",
)


# 6. validate=False 时跳过完整 evaluate()。
train_source = replace_once(
    train_source,
    """        t0 = time.monotonic()
        test_loss, test_acc, test_recall = evaluate(model, test_loader, device, pool_kernel_um=pool_kernel_um)
        test_time = time.monotonic() - t0

        score = test_acc * test_recall
        is_best = score >= best_score

        if is_best:
            best_score = score
            # Normalise any DataParallel "unet.module." prefix to "unet." so the
            # checkpoint loads on a single GPU (e.g. in the prediction script).
            torch.save(
                {k.replace("unet.module.", "unet.", 1): v for k, v in model.state_dict().items()},
                save_path,
            )
""",
    """        if validate:
            t0 = time.monotonic()
            test_loss, test_acc, test_recall = evaluate(
                model,
                test_loader,
                device,
                pool_kernel_um=pool_kernel_um,
            )
            test_time = time.monotonic() - t0

            score = test_acc * test_recall
            is_best = score >= best_score
        else:
            test_loss = float("nan")
            test_acc = float("nan")
            test_recall = float("nan")
            test_time = 0.0
            score = 0.0
            is_best = True

        if is_best:
            best_score = score
            # Normalize any DataParallel prefix before saving.
            torch.save(
                {
                    k.replace("unet.module.", "unet.", 1): v
                    for k, v in model.state_dict().items()
                },
                save_path,
            )
""",
    "evaluation block",
)


# 7. 训练结束时输出正确的模式说明。
train_source = replace_once(
    train_source,
    """    print(f"\\nBest score (acc*recall): {best_score:.4f}, saved to {save_path}", flush=True)
""",
    """    if validate:
        print(
            f"\\nBest score (acc*recall): {best_score:.4f}, "
            f"saved to {save_path}",
            flush=True,
        )
    else:
        print(
            f"\\nTrain-only mode: latest model saved to {save_path}",
            flush=True,
        )
""",
    "training summary",
)


compile(train_source, str(train_script), "exec")
train_script.write_text(train_source, encoding="utf-8")

import inspect
import train_unet_transformer as trainer

assert Path(trainer.__file__).resolve() == train_script.resolve(), (
    f"Loaded wrong trainer: {trainer.__file__}"
)
assert "validate" in inspect.signature(trainer.train).parameters, (
    "Loaded trainer has no validate parameter. Restart the kernel."
)
assert torch.cuda.is_available(), "CUDA GPU is required"

print("Training script patched and imported.", flush=True)
print("trainer:", trainer.__file__, flush=True)
print("train signature:", inspect.signature(trainer.train), flush=True)
print("torch:", torch.__version__, flush=True)
print("CUDA:", torch.cuda.get_device_name(0), flush=True)
print("secondary weights:", secondary_weights, flush=True)
print("secondary config:", secondary_config, flush=True)

In [ ]:
import numpy as np
import zarr

train_root = competition / "train"
sample = next(
    path for path in sorted(train_root.glob("*.zarr"))
    if path.with_suffix(".geff").exists()
)
dataset = trainer.open_dataset(
    sample, require_tracks=True, load_image=False, downsample=(1, 4, 4)
)
image = zarr.open_group(str(dataset.zarr_path), mode="r")["0"]
frame = np.asarray(image[0, :, ::4, ::4])
assert frame.ndim == 3 and frame.size > 0 and np.isfinite(frame).all()
print("读取成功:", sample.name, "图像:", frame.shape)
print("标注节点数:", len(dataset.tracks.node_attrs()))

In [ ]:
initial_weights = secondary_weights
actual_initial_sha256 = hashlib.sha256(initial_weights.read_bytes()).hexdigest()
expected_initial_sha256 = "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f"
assert actual_initial_sha256 == expected_initial_sha256, "Initial weight SHA256 mismatch"
config = json.loads(initial_weights.with_name("config.json").read_text(encoding="utf-8"))
initial_state = torch.load(initial_weights, map_location="cpu", weights_only=True)
data_dir = competition / "train"
names = sorted(
    path.stem for path in data_dir.glob("*.zarr")
    if path.with_suffix(".geff").exists()
)
assert names, f"No annotated training datasets found in {data_dir}"
split = {"split": 0, "train": names, "test": []}
splits_path = run_dir / "splits.json"
splits_path.write_text(json.dumps([split], indent=2), encoding="utf-8")
print(f"Train: {len(names)}; local validation disabled", flush=True)

In [ ]:
BaseModel = trainer.UNetNodeTransformer
reference = {}
class SecondaryFineTune(BaseModel):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.load_state_dict(initial_state, strict=True)
        reference["teacher"] = copy.deepcopy(self).cuda().eval().requires_grad_(False)
        self.requires_grad_(False)
        for module in (self.unet.decoder_blocks[-1], self.unet.head, self.detect_head): module.requires_grad_(True)
    def train(self, mode=True):
        return super().train(False)
    def encode(self, imgs):
        features, logits = super().encode(imgs)
        if torch.is_grad_enabled():
            with torch.no_grad(): teacher_features, teacher_logits = reference["teacher"].encode(imgs)
            reference["logits"] = iter(teacher_logits)
            reference["feature_loss"] = functional.mse_loss(features.float(), teacher_features.float()) / teacher_features.float().square().mean().clamp_min(1e-6)
        return features, logits
def sparse_detection_loss(logits, coords, mask, neg_weight=0.01):
    """结合标注中心监督、未标注区域蒸馏与共享特征约束。"""
    teacher_prob = next(reference["logits"]).float().sigmoid()
    points = coords[mask].long().clamp_min(0).minimum(torch.tensor(logits.shape[2:], device=logits.device) - 1)
    positive = torch.zeros_like(logits, dtype=torch.bool)
    positive[mask.nonzero()[:, 0], 0, points[:, 0], points[:, 1], points[:, 2]] = True
    positive_loss = functional.softplus(-logits.float())[positive].sum() / positive.sum().clamp_min(1)
    distillation = functional.binary_cross_entropy_with_logits(logits.float(), teacher_prob, reduction="none")
    return positive_loss + 0.5 * distillation[~positive].sum() / (~positive).sum().clamp_min(1) + 0.1 * reference["feature_loss"]

In [ ]:
import numpy as np

SYNTH_ROOT = Path("/kaggle/input/datasets/y3uanm/biohub-synthetic-dataset-jose/biohub_synthetic")
synth_files = sorted((SYNTH_ROOT / "sequences").glob("*.npz"))
assert synth_files, f"No synthetic sequences: {SYNTH_ROOT}"
assert tuple(config["downsample"]) == (1, 4, 4), "This adapter requires downsample=(1,4,4)"
SYNTH_WINDOWS_PER_REAL = 0.5
synth_rng = np.random.default_rng(314159)
sample_counts = {"real": 0, "synthetic": 0}
synth_seen = set()
print("Available synthetic sequences:", len(synth_files))
OriginalDataset = trainer.FrameWindowDataset
OriginalFineTune = SecondaryFineTune
real_detection_loss = sparse_detection_loss
domain = {"synthetic": False}

class MixedWindowDataset(OriginalDataset):
    """完整遍历真实窗口，并增加合成窗口；仅支持 num_workers=0。"""
    def __len__(self):
        real_count = super().__len__()
        return real_count + max(1, int(real_count * SYNTH_WINDOWS_PER_REAL))

    def __getitem__(self, index):
        real_count = super().__len__()
        domain["synthetic"] = index >= real_count
        if not domain["synthetic"]:
            sample_counts["real"] += 1
            return super().__getitem__(index)
        sample_counts["synthetic"] += 1
        if not hasattr(self, "_synth_order"):
            self._synth_order = synth_rng.permutation(len(synth_files))
        slot = index - real_count
        sequence_path = synth_files[int(self._synth_order[slot % len(synth_files)])]
        synth_seen.add(str(sequence_path))
        with np.load(sequence_path, allow_pickle=False) as sample:
            volumes = sample["volumes"].astype(np.float32)
            nodes = sample["nodes"].copy()
            edges = sample["edges"].astype(np.int64).reshape(-1, 2)
        window_size = int(config["window_size"])
        assert 2 <= window_size <= len(volumes)
        start = int(synth_rng.integers(len(volumes) - window_size + 1))
        coords, positions, counts, frame_ids = [], [], [], []
        for offset in range(window_size):
            ids = np.flatnonzero(nodes[:, 0] == start + offset)
            points = nodes[ids, 1:4] / np.array([1, 4, 4], np.float32)
            assert len(ids) and np.isfinite(points).all()
            assert ((points >= 0) & (points < np.array(volumes.shape[1:]))).all()
            coords.append(torch.tensor(points, dtype=torch.float32))
            positions.append(torch.zeros(len(ids), 4 * trainer._POS_EMBED_DIM))
            counts.append(len(ids))
            frame_ids.append(ids)
        targets = []
        for source_ids, target_ids in zip(frame_ids[:-1], frame_ids[1:]):
            source_map = {int(node_id): index for index, node_id in enumerate(source_ids)}
            target_map = {int(node_id): index for index, node_id in enumerate(target_ids)}
            matrix = torch.zeros(len(source_ids), len(target_ids))
            for source_id, target_id in edges:
                if int(source_id) in source_map and int(target_id) in target_map:
                    matrix[source_map[int(source_id)], target_map[int(target_id)]] = 1.0
            targets.append(matrix)
        window = trainer.FrameWindowData(start, window_size, positions, coords, counts, targets)
        batch = trainer.pad_window(window, max(counts))
        low, high = np.quantile(volumes, [0.001, 0.999])
        images = torch.from_numpy((volumes[start:start + window_size] - low) / (high - low + 1e-6))
        return {**batch, "imgs": images.clamp_min(0).half(),
                "image_shape": torch.tensor(volumes.shape),
                "voxel_size": torch.tensor([1.625, 1.625, 1.625]),
                "downsample": torch.tensor([1., 4., 4.])}

class SecondaryFineTune(OriginalFineTune):
    """真实窗口保留教师约束；合成窗口跳过教师推理。"""
    def encode(self, imgs):
        return BaseModel.encode(self, imgs) if domain["synthetic"] else super().encode(imgs)

def sparse_detection_loss(logits, coords, mask, neg_weight=0.01):
    """真实数据使用原损失；合成数据使用中心正例及背景负例。"""
    if not domain["synthetic"]:
        return real_detection_loss(logits, coords, mask, neg_weight)
    points = coords[mask].long()
    positive = torch.zeros_like(logits, dtype=torch.bool)
    positive[mask.nonzero()[:, 0], 0, points[:, 0], points[:, 1], points[:, 2]] = True
    values = logits.float()
    return (functional.softplus(-values)[positive].sum() / positive.sum().clamp_min(1)
            + 0.01 * functional.softplus(values)[~positive].sum() / (~positive).sum().clamp_min(1))

trainer.FrameWindowDataset = MixedWindowDataset
print("Synthetic sequences:", len(synth_files), "| real:synth windows approximately 3:1")

In [ ]:
trainer.UNetNodeTransformer = SecondaryFineTune
trainer.compute_detection_loss = sparse_detection_loss
trainer.WEIGHTS_PATH = run_dir / "weights"
model = trainer.train(
    data_dir=data_dir,
    fold=0,
    splits_file=splits_path,
    method="unet_transformer",
    n_epochs=10,
    lr=1e-5,
    batch_size=1,
    num_workers=0,
    seed=314159,
    unet_out_channels=config["unet_out_channels"],
    unet_layers=config["unet_layers"],
    downsample=tuple(config["downsample"]),
    window_size=config["window_size"],
    det_loss_weight=1.0,
    max_iters=None,
    validate=False,
    augmentations=None,
    data_parallel=False,
    pool_kernel_um=float(config.get("pool_kernel_um", 5.0)),
)
candidate = trainer.WEIGHTS_PATH / "unet_transformer/split_0/edge_predictor_best.pth"
model.load_state_dict(torch.load(candidate, map_location="cpu", weights_only=True), strict=True)
candidate.with_name("config.json").write_text(json.dumps(config, indent=2))
print("Candidate:", candidate, "\nSHA256:", hashlib.sha256(candidate.read_bytes()).hexdigest())

In [ ]:
artifact_name = "biohub-temporal-unet3d-seed314159-ft-v2"
artifact_dir = Path("/kaggle/working") / artifact_name

assert not artifact_dir.exists(), (
    f"Artifact already exists: {artifact_dir}"
)

candidate = (
    trainer.WEIGHTS_PATH
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)

candidate_config = candidate.with_name("config.json")

assert candidate.is_file(), candidate
assert candidate_config.is_file(), candidate_config

artifact_dir.mkdir(parents=True)

# 1. 保存本次实际使用的完整运行仓库。
shutil.copytree(repo_dir, artifact_dir / "repo")

# 2. 保存本次修改后的训练脚本。
shutil.copytree(
    repo_dir / "scripts",
    artifact_dir / "source_scripts",
)

# 3. 保存本次训练产生的新权重。
shutil.copytree(
    trainer.WEIGHTS_PATH,
    artifact_dir / "weights",
)

# 4. 保存父模型，便于下次继续微调。
shutil.copytree(
    initial_weights_root,
    artifact_dir / "initial_weights",
)

# 5. 保存数据记录。
shutil.copy2(
    splits_path,
    artifact_dir / "splits.json",
)

# 6. 保存离线依赖包。
support_wheels = support / "wheels"
assert support_wheels.is_dir(), support_wheels

shutil.copytree(
    support_wheels,
    artifact_dir / "wheels",
)

# 7. 保存依赖清单。
for requirement_name in [
    "requirements-unet-ilp-kaggle-predownload.txt",
    "requirements-unet-ilp.txt",
]:
    requirement_source = support / requirement_name
    assert requirement_source.is_file(), requirement_source
    shutil.copy2(
        requirement_source,
        artifact_dir / requirement_name,
    )

# 8. 记录离线安装命令。
install_command_text = (
    f"{sys.executable} -m pip install "
    "--no-index --no-deps "
    "--force-reinstall "
    "--find-links ./wheels "
    + " ".join(runtime_packages)
    + "\n"
)

(artifact_dir / "kaggle_dependency_install_command.txt").write_text(
    install_command_text,
    encoding="utf-8",
)

# 9. 确认导出的权重与训练结果一致。
exported_weights = (
    artifact_dir
    / "weights"
    / "unet_transformer"
    / "split_0"
    / "edge_predictor_best.pth"
)

exported_config = exported_weights.with_name("config.json")

assert exported_weights.is_file(), exported_weights
assert exported_config.is_file(), exported_config

candidate_sha256 = hashlib.sha256(
    candidate.read_bytes()
).hexdigest()

exported_sha256 = hashlib.sha256(
    exported_weights.read_bytes()
).hexdigest()

assert candidate_sha256 == exported_sha256, (
    f"Exported weight mismatch:\n"
    f"candidate: {candidate_sha256}\n"
    f"exported:  {exported_sha256}"
)

# 10. 生成模型包清单。
manifest = {
    "artifact_name": artifact_name,
    "format_version": 1,
    "model_method": "unet_transformer",
    "weights": (
        "weights/unet_transformer/split_0/"
        "edge_predictor_best.pth"
    ),
    "config": (
        "weights/unet_transformer/split_0/config.json"
    ),
    "splits": "splits.json",
    "sha256": exported_sha256,
    "parent_sha256": actual_initial_sha256,
    "training_mode": "real_sparse_teacher_plus_synthetic_dense",
    "validation": "public_leaderboard_submission",
    "n_epochs": 1,
    "max_iters": None,
}

(artifact_dir / "ARTIFACT_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

# 11. 写入使用说明。
readme_text = f"""# {artifact_name}

This artifact contains a fine-tuned UNet + node-transformer model.

## Model

- Method: unet_transformer
- Weights: weights/unet_transformer/split_0/edge_predictor_best.pth
- Config: weights/unet_transformer/split_0/config.json
- Training mode: all annotated training datasets
- Local validation: disabled
- Validation method: Kaggle public leaderboard submission

## SHA256

- Weights: {exported_sha256}
- Parent weights: {actual_initial_sha256}

## Contents

- repo/
- source_scripts/
- initial_weights/
- weights/
- wheels/
- splits.json
- ARTIFACT_MANIFEST.json
- requirements-unet-ilp-kaggle-predownload.txt
- requirements-unet-ilp.txt
"""

(artifact_dir / "README.md").write_text(
    readme_text,
    encoding="utf-8",
)

print("Artifact:", artifact_dir)
print("Weights:", exported_weights)
print("SHA256:", exported_sha256)
print("Files:")
for path in sorted(artifact_dir.iterdir()):
    print(" ", path.name)